In [1]:
from pathlib import Path
import os
import pandas as pd
import sklearn

base_folder = Path(
    r"C:\Users\bamsa\OneDrive\Desktop\NUS - AI\Copstone Project"
)

part3_folder = base_folder / "capstone_part3"
part3_folder.mkdir(exist_ok=True)
os.chdir(part3_folder)

raw_csv = (
    base_folder / "capstone_part2"
    / "data" / "Metro_Interstate_Traffic_Volume.csv"
)

df = pd.read_csv(raw_csv, keep_default_na=False)

print("작업 폴더:", Path.cwd())
print("데이터 크기:", df.shape)
print("scikit-learn 버전:", sklearn.__version__)

df.head()

작업 폴더: C:\Users\bamsa\OneDrive\Desktop\NUS - AI\Copstone Project\capstone_part3
데이터 크기: (48204, 9)
scikit-learn 버전: 1.7.2


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [2]:
import logging

logger = logging.getLogger("part3_preparation")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            part3_folder / "part3.log",
            mode="a",
            encoding="utf-8",
        ),
    ],
    force=True,
)

# 원본 데이터는 유지하고 복사본으로 작업합니다.
data = df.copy()

data["date_time"] = pd.to_datetime(
    data["date_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise",
)

# 모든 열의 값이 동일한 중복만 제거합니다.
before = len(data)
data = data.drop_duplicates().copy()

logger.warning(
    "Removed exact duplicate rows=%d",
    before - len(data),
)

data = data.sort_values("date_time").reset_index(drop=True)

# 고유한 시각을 기준으로 분리합니다.
timestamps = data["date_time"].drop_duplicates().sort_values()
train_end = timestamps.iloc[int(len(timestamps) * 0.70)]
test_start = timestamps.iloc[int(len(timestamps) * 0.85)]

train = data.loc[data["date_time"] < train_end].copy()

validation = data.loc[
    (data["date_time"] >= train_end)
    & (data["date_time"] < test_start)
].copy()

test = data.loc[data["date_time"] >= test_start].copy()

# 분리 구간이 겹치지 않는지 확인합니다.
assert train["date_time"].max() < validation["date_time"].min()
assert validation["date_time"].max() < test["date_time"].min()

for name, subset in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    logger.info(
        "%s | rows=%d | unique timestamps=%d | start=%s | end=%s",
        name,
        len(subset),
        subset["date_time"].nunique(),
        subset["date_time"].min(),
        subset["date_time"].max(),
    )

summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(subset),
        "unique_hours": subset["date_time"].nunique(),
        "start": subset["date_time"].min(),
        "end": subset["date_time"].max(),
    }
    for name, subset in [
        ("train", train),
        ("validation", validation),
        ("test", test),
    ]
])

summary

2026-10-10 22:31:13,082 | WARNING | part3_preparation | Removed exact duplicate rows=17
2026-10-10 22:31:13,153 | INFO | part3_preparation | train | rows=33497 | unique timestamps=28402 | start=2012-10-02 09:00:00 | end=2017-05-10 04:00:00
2026-10-10 22:31:13,156 | INFO | part3_preparation | validation | rows=7274 | unique timestamps=6086 | start=2017-05-10 05:00:00 | end=2018-01-19 14:00:00
2026-10-10 22:31:13,158 | INFO | part3_preparation | test | rows=7416 | unique timestamps=6087 | start=2018-01-19 15:00:00 | end=2018-09-30 23:00:00


,split,rows,unique_hours,start,end
0,train,33497,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
1,validation,7274,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
2,test,7416,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


In [3]:
import numpy as np
import pandas as pd

weather_columns = [
    "temp", "rain_1h", "snow_1h", "clouds_all"
]

# 이상값을 결측값으로 바꾸기
def mark_invalid_weather(frame):
    result = frame.copy()

    for col in weather_columns:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

    result.loc[result["temp"] <= 0, "temp"] = np.nan

    result.loc[
        (result["rain_1h"] < 0) | (result["rain_1h"] > 9000),
        "rain_1h"
    ] = np.nan

    result.loc[result["snow_1h"] < 0, "snow_1h"] = np.nan

    result.loc[
        ~result["clouds_all"].between(0, 100),
        "clouds_all"
    ] = np.nan

    return result


# 중앙값은 학습 데이터에서만 계산
train_checked = mark_invalid_weather(train)

monthly_medians = train_checked.groupby(
    train_checked["date_time"].dt.month
)[weather_columns].median()

overall_medians = train_checked[weather_columns].median()

if overall_medians.isna().any():
    raise ValueError("학습 데이터에서 중앙값을 계산할 수 없는 열이 있습니다.")


# 학습 데이터의 기준을 모든 구간에 적용
def clean_weather(frame, split_name):
    result = mark_invalid_weather(frame)
    month = result["date_time"].dt.month

    for col in weather_columns:
        missing_count = int(result[col].isna().sum())

        result[col] = (
            result[col]
            .fillna(month.map(monthly_medians[col]))
            .fillna(overall_medians[col])
        )

        if missing_count:
            logger.warning(
                "%s: filled %d invalid/missing values in %s",
                split_name, missing_count, col
            )

    logger.info(
        "%s: weather cleaning completed, rows=%d",
        split_name, len(result)
    )
    return result


train_clean = clean_weather(train, "train")
validation_clean = clean_weather(validation, "validation")
test_clean = clean_weather(test, "test")

# 결과 확인: 모든 칸이 0이면 결측값 처리 완료
pd.DataFrame({
    "train_missing": train_clean[weather_columns].isna().sum(),
    "validation_missing": validation_clean[weather_columns].isna().sum(),
    "test_missing": test_clean[weather_columns].isna().sum()
})

2026-10-10 22:36:28,809 | WARNING | part3_preparation | train: filled 10 invalid/missing values in temp
2026-10-10 22:36:28,813 | WARNING | part3_preparation | train: filled 1 invalid/missing values in rain_1h
2026-10-10 22:36:28,818 | INFO | part3_preparation | train: weather cleaning completed, rows=33497
2026-10-10 22:36:28,834 | INFO | part3_preparation | validation: weather cleaning completed, rows=7274
2026-10-10 22:36:28,848 | INFO | part3_preparation | test: weather cleaning completed, rows=7416


,train_missing,validation_missing,test_missing
temp,0,0,0
rain_1h,0,0,0
snow_1h,0,0,0
clouds_all,0,0,0
